In [ ]:
#@title 1. Upload RPF zip files
from google.colab import files

print("a. upload RPF zip file for Model A")
proteinAUploaded = files.upload()
proteinAFileName = list(proteinAUploaded.keys())[0]
print("Model A RPF zip file uploaded: ", proteinAFileName)

print("b. upload RPF zip file for Model B")
proteinBUploaded = files.upload()
proteinBFileName = list(proteinBUploaded.keys())[0]
print("Model B RPF zip file uploaded: ", proteinBFileName)


a. upload RPF zip file for Model A


Saving rpf_CDK2AP1_state2_20251120_11_30_32.zip to rpf_CDK2AP1_state2_20251120_11_30_32.zip
Model A RPF zip file uploaded:  rpf_CDK2AP1_state2_20251120_11_30_32.zip
b. upload RPF zip file for Model B


Saving rpf_CDK2AP1_state1_20251120_11_27_32.zip to rpf_CDK2AP1_state1_20251120_11_27_32.zip
Model B RPF zip file uploaded:  rpf_CDK2AP1_state1_20251120_11_27_32.zip


In [ ]:
#@title 2. Extract *.match and *.QR from the Zip files
from zipfile import ZipFile
import gzip

with ZipFile(proteinAFileName, 'r') as zipAObj:
  zipAObj.extractall()

with ZipFile(proteinBFileName, 'r') as zipBObj:
  zipBObj.extractall()

#cat all match files to all_match
AFolder = zipAObj.namelist()[0]
BFolder = zipBObj.namelist()[0]

#print(AFolder, BFolder)
#Aall_match
print("The following *.match files are found:")
Aall_match = ''
AQR = ''
for a in zipAObj.namelist():
  if a.startswith('__MACOSX/'):
    continue
  if a.endswith('match.gz'):
    print(a)
    with gzip.open(a, 'rt') as matchObj:
      match_content = matchObj.read()
      Aall_match += match_content
  elif a.endswith('QR'):
    AQR = a

#Ball_match
Ball_match = ''
BQR = ''
for b in zipBObj.namelist():
  if b.startswith('__MACOSX/'):
    continue
  if b.endswith('match.gz'):
    print(b)
    with gzip.open(b, 'rt') as matchObj:
      match_content = matchObj.read()
      Ball_match += match_content
  elif b.endswith('QR'):
    BQR = b

print("\nThe following *.QR files are found:")
print(AQR)
print(BQR)

The following *.match files are found:
rpf_CDK2AP1_state2_20251120_11_30_32/c13ali_match.gz
rpf_CDK2AP1_state2_20251120_11_30_32/n15noesy_match.gz
rpf_CDK2AP1_state1_20251120_11_27_32/c13ali_match.gz
rpf_CDK2AP1_state1_20251120_11_27_32/n15noesy_match.gz

The following *.QR files are found:
rpf_CDK2AP1_state2_20251120_11_30_32/CDK2AP1_state2_DP.QR
rpf_CDK2AP1_state1_20251120_11_27_32/CDK2AP1_state1_DP.QR


In [ ]:
#@title 3. Double Recall Analysis
import re, sys

#@markdown Number of Identical Chains in the Sequence: (e.g. homodimer's N_chain = 2)
N_chain  = 2 #@param {type:"number"}
def getDiff(l1, l2_sub):
    diffList=[]
    for line1 in l1:
        #print(line)
        if not line1.strip():
            continue

        line12 = line1.strip()
        #print("reading+" + line12.strip() + "+")
        line12_sub = re.sub(r'[\s+|\(|\)]', "", line12)

        inf2 = re.findall(line12_sub, l2_sub)
        #print("inf2" + ' '.join(inf2))
        if (not inf2):
            diffList.append(line1.strip())
    return diffList


def getH1H2(list, all_f3):
    hx1=[]
    h1h2=[]

    for line in list:
        if not line.strip():
            continue

        #line2 = "-> peak:  5314 (hx1)7.105 (hx2)7.724 (x1)133.123  5.1299 (label)5314"

        line2 = line.strip()
        line2_sub = re.sub(r'[\s+|\(|\)|\-|\>]', "", line2)
        #print("debug: reading" + line2_sub)

        uniqueHX1Match = 0
        for f3line in all_f3:
            if not f3line.strip():
                continue

            line3 = f3line.strip()
            line3_sub = re.sub(r'[\s+|\(|\)]', "", line3)
            #print(line3_sub+"\n"+line2_sub)
            inf2 = re.findall(line2_sub, line3_sub)
            if (inf2):

                #print("debug: line2 " + line2_sub)
                #print("debug: line3 " + line3_sub)
                #if re.search(r'poss 2*', f3line): #dimer
                #my_regex = r'poss ' + re.escape(N_chain_str) + r"!\d"
                if re.search(rf'poss {N_chain} \*', f3line): #dimer
                  uniqueHX1Match = 1

                hx11 = ''
                matchlist = f3line.split("\n")
                for hline in matchlist:

                    #print(hline)

                    atomH1re = re.compile(r'^HX1: atom\:\s+\d+\s+(\d+[A-Z]\-[A-Z]+\d*)\s+')
                    atom1 = atomH1re.findall(hline)

                    #print(atom1)
                    if (atom1):
                        #hx1.append(atom1[0])
                        hx11 = atom1[0]
                    else:
                        #mid distance < 5 ang
                        disre = re.compile(r'(\d+\.*\d*)\(mid\)')
                        dis = disre.findall(hline)

                        #print(dis)
                        #if (dis):
                        #    print(float(dis[0]))
                        if (dis and float(dis[0]) <= 5): #explained
                            #print("f3line " + f3line)
                            atomH2re = re.compile(r'^   atom\:\s+\d+\s+(\d+[A-Z]\-[A-Z]+\d*)\s+')
                            atom2 = atomH2re.findall(hline)

                            #print(atom2)
                            if (hx11 and atom2[0]):
                                h1h2.append([hx11, atom2[0]])
                                print(uniqueHX1Match, hx11, atom2[0], line2) #output
                                if (uniqueHX1Match):
                                  #print("f3line " + f3line)
                                  hx1.append(hx11)
                            #list.insert(hx1, atom)
                break;
    return (hx1, h1h2)

with open(AQR, 'r') as f1:
  all_AQR = f1.read()

with open(BQR, 'r') as f2:
  all_BQR = f2.read()

all_AQR_sub = re.sub(r'[\s+|\(|\)]', "", all_AQR)
all_BQR_sub = re.sub(r'[\s+|\(|\)]', "", all_BQR)

lstA = all_AQR.splitlines()
lstB = all_BQR.splitlines()

lstA_notlstB = getDiff(lstA, all_BQR_sub)
lstB_notlstA = getDiff(lstB, all_AQR_sub)

print("Protein A Recall Violations, but <5 in Protein B")
(lstAHX1, lstAH1H2) = getH1H2(lstA_notlstB, Ball_match.split("#\n#\n"))
print(lstAHX1, lstAH1H2)
print("Protein B Recall Violations, but <5 in Protein A")
(lstBHX1, lstBH1H2) = getH1H2(lstB_notlstA, Aall_match.split("#\n#\n"))
print(lstBHX1, lstBH1H2)

Protein A Recall Violations, but <5 in Protein B
0 10I-HG13 13L-HG -> peak:     7 (hx1)1.132 (hx2)2.329 (x1)29.79  51010 (label)7
0 210I-HG13 213L-HG -> peak:     7 (hx1)1.132 (hx2)2.329 (x1)29.79  51010 (label)7
1 42V-HG1 204A-H -> peak:    22 (hx1)1.063 (hx2)7.915 (x1)23.576  55450 (label)22
1 242V-HG1 4A-H -> peak:    22 (hx1)1.063 (hx2)7.915 (x1)23.576  55450 (label)22
1 220T-HA 228M-HG2 -> peak:    43 (hx1)4.088 (hx2)2.075 (x1)69.319  87570 (label)43
1 220T-HA 228M-HG3 -> peak:    43 (hx1)4.088 (hx2)2.075 (x1)69.319  87570 (label)43
1 201S-HB2 49T-HG2 -> peak:    51 (hx1)4.093 (hx2)1.309 (x1)63.552  47354 (label)51
1 201S-HB3 49T-HG2 -> peak:    53 (hx1)3.982 (hx2)1.314 (x1)63.544  51474 (label)53
1 13L-HG 10I-HG13 -> peak:    63 (hx1)2.347 (hx2)1.118 (x1)25.596  53398 (label)63
1 213L-HG 210I-HG13 -> peak:    63 (hx1)2.347 (hx2)1.118 (x1)25.596  53398 (label)63
1 17I-HA 231L-HG -> peak:   103 (hx1)3.437 (hx2)1.611 (x1)67.309  78017 (label)103
1 217I-HA 31L-HG -> peak:   103 (hx1)

In [ ]:
#@title 4. Make Plots

#@markdown ####4.1. Definition<br>
#@markdown H1 dimension - the H dimension with attached C or N<br>
#@markdown H2 dimension - the indirection H dimension<br>
#@markdown H1 Match - the X-axis for residue with matches on H1 dimension<br>
#@markdown H2 Match - the Y-axis for residue with matches on H2 dimension<br>
#@markdown Unique H1 - the number of H1s that have only one H matched from the chemical shift list<br>

#@markdown ####4.2. Plots<br>
#@markdown Top plot - a bar chart counting number of unique H1 NOESY peaks that can be exclusively explained by one model and not the other<br>
#@markdown Bottom plot - a contact map where a NOESY peak based on H-H distance from the RPF zip file for residue i to j exclusively explained by one model and not the other<br>

#@markdown ####4.3. Settings
import plotly.graph_objects as go

Length = 270 #@param {type:"number"}
#@markdown - the length for the "H1/H2 Match" axes
MaxH1 = 5 #@param {type: "number"}
#@markdown - the length for "unique H1" axis
Model_A_Name = 'state2' #@param {type:"string"}
Model_A_Color = 'blue' #@param {type: "string"}

Model_B_Name = 'state1' #@param {type:"string"}
Model_B_Color = 'orange' #@param {type:"string"}

#H1plot
lstAH1Index = []
for h1 in lstAHX1:
  rindex = re.findall(r'^\d+', h1)
  n = int(rindex[0])
  if n > 200: #one chain
    n = n-200
  lstAH1Index.append(n)

lstBH1Index = []
for h1 in lstBHX1:
  rindex = re.findall(r'^\d+', h1)
  n = int(rindex[0])
  if n > 200:
    n = n-200
  lstBH1Index.append(n)

#print(lstBH1Index, lstAH1Index)
lstsetB = set(lstBH1Index)
lstsetA = set(lstAH1Index)

#print(lstsetA, lstsetB)

#2D plot
Bplot = []
h1Index = []

Blongrange = 0
BlongrangeHX1 = 0
Alongrange = 0
AlongrangeHX1 = 0

for a in lstAH1H2:
  h1Index = re.findall(r'^\d+', a[0])
  h2Index = re.findall(r'^\d+', a[1])

  h1n = int(h1Index[0])
  h2n = int(h2Index[0])
  if (h1n > 200 and h2n > 200): #same chain
    h1n = h1n-200
    h2n = h2n-200

  if (h1n > 200 and h2n < 200): #same chain
    h1n = h1n-200
    h2n = h2n+200

#  print(h1Index[0], h2Index[0])
#  print(h1n, h2n)

  Bplot.append((h1n, h2n))
  if (abs(h1n-h2n) >= 5):
    Blongrange +=1
    #print(h1Index[0], lstsetA)
    if h1n in lstsetA:
      BlongrangeHX1 +=1
  #AH1[h1Index[0]] = AH1[h1Index[0]]+1

Aplot = []
h2Index = []
for b in lstBH1H2:
  h1Index = re.findall(r'^\d+', b[0])
  h2Index = re.findall(r'^\d+', b[1])

  h1n = int(h1Index[0])
  h2n = int(h2Index[0])
  if (h1n > 200 and h2n > 200): #same chain
    h1n = h1n-200
    h2n = h2n-200

  if (h1n > 200 and h2n < 200): #same chain
    h1n = h1n-200
    h2n = h2n+200

  #print(h1Index[0], h2Index[0])
  Aplot.append((h1n, h2n))
  if (abs(h1n-h2n) >= 5):
    Alongrange +=1
    if h1n in lstsetB:
      AlongrangeHX1 +=1
  #AH1[h1Index[0]] = AH1[h1Index[0]]+1

#print(lstBH1Index)
#stat

print("Number of H-Hpairs explained by ", Model_B_Name, ":", len(lstAH1H2), "longrange:", Blongrange, "(unique H1:", BlongrangeHX1,")")
print("Number of H-Hpairs explained by ", Model_A_Name, ":", len(lstBH1H2), "longrange:", Alongrange, "(unique H1:", AlongrangeHX1,")")


trace1 = go.Scatter(
      x=list(zip(*Aplot))[0],
      y=list(zip(*Aplot))[1],
      name = Model_A_Name,
      #opacity = 0.5,
      mode="markers",
      marker = dict(
          size = 5,
          color=Model_A_Color
      )
)

#print(Aplot)
#print(Bplot)
trace2 = go.Scatter(
    x=list(zip(*Bplot))[0],
    y=list(zip(*Bplot))[1],
    name = Model_B_Name,
    #opacity = 0.5,
    mode="markers",
    marker = dict(
       size = 5,
       color=Model_B_Color
    )
)

trace3 = go.Histogram(
    x=lstBH1Index,
    yaxis="y2",
    xbins={"size": 1},
    name = Model_A_Name,
    showlegend=False,
    marker = dict(
      #size = 25,
      color=Model_A_Color
    )
 )
trace4 = go.Histogram(
    x=lstAH1Index,
    yaxis="y2",
    xbins={"size": 1},
    name = Model_B_Name,
    showlegend=False,
    marker = dict(
        #size = 25,
        color=Model_B_Color
    )
)

data= [trace1, trace2, trace3, trace4]

layout = go.Layout(
    margin=dict(l=20, r=20, t=20, b=20),
    autosize=False,
    height=600,
    xaxis = dict(
      range = [-5,Length+5],
      autorange=False,
      title = 'H1 Match',
      showgrid = True,
      zeroline = True,
      showline = True,
      dtick = 10,
      tick0 = 0,
     ),
    yaxis = dict(
      range = [-5,Length+5],
      autorange=False,
      domain = [0,0.85],
      title = 'H2 Match',
      showgrid = True,
      zeroline = True,
      showline = True,
      dtick = 10,
      tick0 = 0,
    ),
    yaxis2 = dict(
      range = [0,MaxH1],
      domain=[0.85,1],
      title = 'Unique H1',
      zeroline = True,
    )
    #paper_bgcolor="LightSteelBlue"
)

fig = go.Figure(data=data, layout=layout)
fig.show()


Number of H-Hpairs explained by  state1 : 22 longrange: 16 (unique H1: 14 )
Number of H-Hpairs explained by  state2 : 23 longrange: 3 (unique H1: 1 )
